In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

# Define the path to the zip file and the extraction directory
zip_file_path = '/content/drive/MyDrive/UCSD_Anomaly_Dataset.zip'
extract_path = '/content/UCSD_Anomaly'

# Extract the zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed.")

Extraction completed.


In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from einops import rearrange, repeat
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm import tqdm
from tqdm.auto import tqdm

C:\Users\mayan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("Is CUDA available?", torch.cuda.is_available())
print("CUDA device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
print("CUDA version:", torch.version.cuda)
print("Number of CUDA devices:", torch.cuda.device_count())

Is CUDA available? True
CUDA device name: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA version: 12.4
Number of CUDA devices: 1


In [4]:
# Define the dataset class
class UCSDDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.frames = []
        self.load_frames()

    def load_frames(self):
        for folder in os.listdir(self.root_dir):
            folder_path = os.path.join(self.root_dir, folder)
            if os.path.isdir(folder_path) and (folder.startswith('Train') or folder.startswith('Test')):
                for frame_name in os.listdir(folder_path):
                    if frame_name.endswith('.tif'):
                        self.frames.append(os.path.join(folder_path, frame_name))
        print(f"Loaded {len(self.frames)} frames from {self.root_dir}")

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        frame_path = self.frames[idx]
        try:
            image = Image.open(frame_path).convert("RGB")
        except (OSError, IOError) as e:
            print(f"Error loading image {frame_path}: {e}")
            return self.__getitem__((idx + 1) % len(self.frames))
        if self.transform:
            image = self.transform(image)
        return image

# Transformations for the images
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomResizedCrop((128, 128), scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets and dataloaders
train_dataset = UCSDDataset('UCSD_Anomaly_Dataset_2/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Train', transform=transform)
test_dataset = UCSDDataset('UCSD_Anomaly_Dataset_2/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Test', transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,  # Increased from 4
    shuffle=True,
    num_workers=0,  # Adjusted based on CPU cores
    pin_memory=True,
    persistent_workers=False  # Changed to False to reduce memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,  # Reduced from 32
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Check if data loading works correctly
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Loaded 6800 frames from UCSD_Anomaly_Dataset_2/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Train
Loaded 7200 frames from UCSD_Anomaly_Dataset_2/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Test
Number of training samples: 6800
Number of testing samples: 7200


In [5]:
class SelectiveScan(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        
        # Projects
        self.in_proj = nn.Linear(d_model, d_model * expand)
        self.conv = nn.Conv1d(d_model * expand, d_model * expand, d_conv,
                            padding=d_conv-1, groups=d_model * expand)
        
        # State space parameters with proper initialization
        self.A = nn.Parameter(torch.randn(d_model * expand) / d_state**0.5)
        self.B = nn.Parameter(torch.randn(d_model * expand, d_state) / d_state**0.5)
        self.C = nn.Parameter(torch.randn(d_model * expand, d_state) / d_state**0.5)
        self.D = nn.Parameter(torch.randn(d_model * expand) / d_model**0.5)
        
        # Output projection
        self.out_proj = nn.Linear(d_model * expand, d_model)

    def forward(self, x):
        # Get input dimensions
        batch_size, H, W, C = x.shape
        L = H * W
        expanded_dim = self.d_model * self.expand

        # Reshape input to sequence
        x = rearrange(x, 'b h w c -> b (h w) c')
        
        # Input projection
        x = self.in_proj(x)  # Shape: [batch_size, L, C*expand]
        
        # Convolution branch
        x_conv = rearrange(x, 'b l (c e) -> b (c e) l', e=self.expand)
        x_conv = self.conv(x_conv)[..., :L]
        x_conv = rearrange(x_conv, 'b (c e) l -> b l (c e)', e=self.expand)
        
        # State space branch - Vectorized implementation
        x_ssm = x
        x_ssm_t = x_ssm.transpose(0, 1)  # [L, B, C*expand]
        
        # Initialize state
        state = torch.zeros(
            batch_size, 
            expanded_dim, 
            self.d_state, 
            device=x.device, 
            dtype=x.dtype
        )
        
        # Precompute exponential of A
        A_exp = torch.exp(self.A).view(1, -1, 1)  # [1, C*expand, 1]
        
        # Compute state updates efficiently
        outputs = []
        for t in range(L):
            # Update state
            state = state * A_exp
            state = state + torch.einsum('bi,ij->bij', x_ssm_t[t], self.B)
            
            # Compute output for current timestep
            out_t = torch.einsum('bij,ij->bi', state, self.C)
            outputs.append(out_t)
        
        # Stack outputs and add D term
        y = torch.stack(outputs, dim=1)  # [B, L, C*expand]
        y = y + self.D.view(1, 1, -1) * x_ssm
        
        # Combine branches
        x = x_conv + y
        
        # Output projection
        x = self.out_proj(x)
        
        # Reshape back to image
        x = rearrange(x, 'b (h w) c -> b h w c', h=H)
        
        return x

In [6]:
class VisionMambaBlock(nn.Module):
    def __init__(self, dim, d_state=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.selective_scan = SelectiveScan(dim, d_state)  # Remove checkpoint here
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Linear(dim * 2, dim)
        )

    def forward(self, x):
        # Apply checkpoint in forward pass
        x = x + torch.utils.checkpoint.checkpoint(self.selective_scan, self.norm1(x), use_reentrant=False)
        x = x + self.mlp(self.norm2(x))
        return x

In [7]:
class VisionMamba(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, dim=256, depth=12, d_state=16):
        super().__init__()

        # Input embedding
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_channels, dim, kernel_size=7, stride=1, padding=3),
            nn.GELU()
        )

        # Mamba blocks
        self.blocks = nn.ModuleList([
            VisionMambaBlock(dim, d_state=d_state)
            for _ in range(depth)
        ])

        # Output head
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Conv2d(dim, out_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        # Input embedding
        x = self.patch_embed(x)
        B, C, H, W = x.shape
        x = rearrange(x, 'b c h w -> b h w c')

        # Process through Mamba blocks
        for block in self.blocks:
            x = block(x)

        # Output processing
        x = self.norm(x)
        x = rearrange(x, 'b h w c -> b c h w')
        x = self.head(x)

        return x

In [8]:
# Training configuration
def get_model_config():
    return {
        'in_channels': 3,
        'out_channels': 3,
        'dim': 32,  # Reduced from 256
        'depth': 4,  # Reduced from 12
        'd_state': 4  # Reduced from 16
    }

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionMamba(**get_model_config()).to(device)

# Initialize weights
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
    elif isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
    if hasattr(m, 'bias') and m.bias is not None:
        nn.init.zeros_(m.bias)

# Apply weight initialization
model.apply(init_weights)
for param in model.parameters():
    param.requires_grad = True

criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

import os

# Create directory for saving models
save_dir = './saved_models'
os.makedirs(save_dir, exist_ok=True)

# Training loop
num_epochs = 20
train_losses = []
best_loss = float('inf')
patience = 5
trigger_times = 0

for epoch in range(num_epochs):
    torch.cuda.empty_cache()
    model.train()
    train_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)
    for data in pbar:
        inputs = data.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Update progress bar with current loss
        train_loss += loss.item() * inputs.size(0)
        current_loss = train_loss / ((pbar.n + 1) * inputs.size(0))
        pbar.set_postfix({'loss': f'{current_loss:.4f}'})

    # Compute average loss for epoch
    train_loss = train_loss / len(train_loader.dataset)
    train_losses.append(train_loss)

    print(f"\nEpoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}")
    scheduler.step()

    # Early stopping check
    if train_loss < best_loss:
        best_loss = train_loss
        trigger_times = 0
        best_model_path = os.path.join(save_dir, 'best_model.pth')
        save_model(model, optimizer, scheduler, epoch, train_loss, best_model_path)
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping triggered!")
            break

# Save the model at the end of training
final_model_path = os.path.join(save_dir, 'final_model.pth')

def save_model(model, optimizer, scheduler, epoch, loss, final_model_path):
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch,
        'loss': loss,
        'model_config': get_model_config()  # Save config for reconstruction
    }, final_model_path)

print(f"Final model saved to {final_model_path}")

Epoch 1/20:   0%|                                      | 0/213 [00:00<?, ?it/s]C:\Users\mayan\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
Epoch 1/20:  27%|███▏        | 57/213 [11:01:00<30:09:04, 695.80s/it, loss=nan]


KeyboardInterrupt: 

In [ ]:
# Add this after creating datasets
print("Checking first few frames:")
print(train_dataset.frames[:5])

In [ ]:
from PIL import Image
def compute_reconstruction_error(model, data_loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for data in data_loader:
            inputs = data.to(device)
            outputs = model(inputs)
            error = F.mse_loss(outputs, inputs, reduction='none').mean(dim=[1,2,3])
            errors.extend(error.cpu().numpy())
    return np.array(errors)

# Compute reconstruction errors on the test set
test_errors = compute_reconstruction_error(model, test_loader)

# Assuming ground truth labels for anomalies (0 for normal, 1 for anomaly)
# For this example, we will create a dummy ground truth labels array.
# Replace this with actual labels if available.
test_labels = np.zeros(len(test_errors))
# Mark some frames as anomalous for demonstration purposes
test_labels[len(test_labels) // 2:] = 1

# Calculate ROC-AUC
roc_auc = roc_auc_score(test_labels, test_errors)
print(f"ROC-AUC: {roc_auc:.4f}")

# Calculate EER
fpr, tpr, thresholds = roc_curve(test_labels, test_errors)
fnr = 1 - tpr
eer_threshold = thresholds[np.nanargmin(np.absolute(fnr - fpr))]
eer = fpr[np.nanargmin(np.absolute(fnr - fpr))]
print(f"EER: {eer:.4f}")